# Meningioma Atypia Calculator — Build & Validation

Notebook for turning the **multivariable logistic model** from the association pipeline
into a clinician-facing risk calculator, then validating it against the cohort.

Prototype here first; stable logic moves to `atypier_calculator.py` once tested.

| Stage | Purpose |
|-------|---------|
| Load outputs | Cleaned cohort + pooled regression coefficients |
| Build | Score new MRI inputs → P(high-grade WHO 2/3) |
| Validate | Compare calculator vs observed `high_grade` on held-in cohort rows |
| Export | Refactor working code into `atypier_calculator.py` (+ Streamlit later) |

**Inputs** (from `meningioma.ipynb` pipeline):

```
output/cleaning/cleaned.csv
output/inferential/tables/inferential_summary.csv
```

**Target:** `high_grade` — WHO grade 2 or 3 vs grade 1.


## 0. Setup

In [1]:
import pandas as pd
from pathlib import Path

OUTPUT_ROOT = Path("output")
CLEANED_PATH = OUTPUT_ROOT / "cleaning" / "cleaned.csv"
MODEL_PATH = OUTPUT_ROOT / "inferential" / "tables" / "inferential_summary.csv"

## 1. Load tables


In [2]:
df = pd.read_csv(CLEANED_PATH)
model = pd.read_csv(MODEL_PATH)

print(f"df: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"model: {model.shape[0]} rows × {model.shape[1]} cols")

df: 366 rows × 40 cols
model: 6 rows × 14 cols


## 2. Calculator

In [3]:
display(model)
model.predictor_col.unique()

,target,predictor_col,or,or_ci_lo,or_ci_hi,coef,se,p,df,n_models,intercept_coef,intercept_or,z_mu,z_sd
0,high_grade,max_diameter_cm,1.36,1.06,1.74,0.31,0.13,0.015,∞,1,-0.41,0.662,4.04,1.49
1,high_grade,capsular_enhancement,0.99,0.45,2.17,-0.01,0.40,0.973,∞,1,-0.41,0.662,NaN,NaN
2,high_grade,cystic_component,2.52,1.40,4.53,0.92,0.30,0.002,∞,1,-0.41,0.662,NaN,NaN
3,high_grade,cortical_destruction,2.26,1.04,4.89,0.81,0.39,0.039,∞,1,-0.41,0.662,NaN,NaN
4,high_grade,t1_hypointensity,0.45,0.17,1.20,-0.80,0.50,0.111,∞,1,-0.41,0.662,NaN,NaN
5,high_grade,adc_value,0.76,0.58,0.99,-0.28,0.13,0.041,∞,1,-0.41,0.662,0.84,0.13


<ArrowStringArray>
[     'max_diameter_cm', 'capsular_enhancement',     'cystic_component',
 'cortical_destruction',     't1_hypointensity',            'adc_value']
Length: 6, dtype: str

In [4]:
import numpy as np

def high_grade_probability(x, coef):
    logit = (
        coef["intercept_coef"]
        + coef["cystic_component"] * x["cystic_component"]
        + coef["cortical_destruction"] * x["cortical_destruction"]
        + coef["max_diameter_cm"] * x["max_diameter_cm"]
        + coef["t1_hypointensity"] * x["t1_hypointensity"]
        + coef["adc_value"] * x["adc_value"]
        + coef["capsular_enhancement"] * x["capsular_enhancement"]
    )
    return 1 / (1 + np.exp(-logit))

In [5]:
df[~df.high_grade][["max_diameter_cm",
    "capsular_enhancement",
    "cystic_component",
    "cortical_destruction",
    "t1_hypointensity",
    "adc_value"]].head()

,max_diameter_cm,capsular_enhancement,cystic_component,cortical_destruction,t1_hypointensity,adc_value
0,4.9,True,False,False,True,0.88
1,2.8,True,False,False,True,0.94
3,3.7,True,False,False,True,1.20
4,2.9,NaN,NaN,NaN,NaN,0.93
5,2.8,True,False,False,True,0.81


In [6]:
def _safe_z_denominator(sd: float) -> float:
    if pd.isna(sd) or not np.isfinite(sd) or sd == 0:
        return 1.0
    return float(sd)

def model_params_from_table(model: pd.DataFrame) -> dict:
    params = {"intercept_coef": float(model["intercept_coef"].iloc[0])}
    for _, row in model.iterrows():
        p = row["predictor_col"]
        params[p] = {
            "coef": float(row["coef"]),
            "z_mu": float(row["z_mu"]) if pd.notna(row["z_mu"]) else None,
            "z_sd": float(row["z_sd"]) if pd.notna(row["z_sd"]) else None,
        }
    return params

def high_grade_probability(patient: dict, params: dict) -> float:
    logit = params["intercept_coef"]
    for predictor, spec in params.items():
        if predictor == "intercept_coef":
            continue
        x = patient[predictor]
        if spec["z_mu"] is not None:
            x = (x - spec["z_mu"]) / _safe_z_denominator(spec["z_sd"])
        else:
            x = int(x)
    return float(1 / (1 + np.exp(-logit)))

params = model_params_from_table(model)

patient = {
    "max_diameter_cm": 2.8,
    "capsular_enhancement": 1,
    "cystic_component": 0,
    "cortical_destruction": 0,
    "t1_hypointensity": 1,
    "adc_value": 0.94,
}


high_grade_probability(patient, params)

0.3989121211516302